# T2.4 - View Definitions
Owner: D (El Dib Yehea)
Creates the ML pipeline views in DBRepo via the REST API.

In [7]:
import requests
import os
from dotenv import load_dotenv
from requests.auth import HTTPBasicAuth

load_dotenv()

BASE_URL    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = os.getenv("DBREPO_USERNAME")
PASSWORD    = os.getenv("DBREPO_PASSWORD")
DATABASE_ID = "82c19b39-246c-4409-b25c-8baf3a158a70"
TABLE_ID    = "a5b0155a-9fc6-48f6-8b24-1f591dbf5cad"


In [8]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    db = response.json()
    print(f"Connection successful")
    print(f"Database name : {db.get('name')}")
    print(f"Database ID   : {db.get('id')}")
else:
    print(f"Connection failed with status {response.status_code}")
    print(response.text[:200])

Connection successful
Database name : uk-collision-severity-prediction-main
Database ID   : 82c19b39-246c-4409-b25c-8baf3a158a70


In [9]:
VIEWS = [
    {
        "name": "collision_ml_features",
        "purpose": "Clean feature table matching exactly the input expected by 01_load_data.py. Contains the 15 features and label used for ML training. Primary source for T2.6 API reimplementation.",
        "body": {
            "name": "collision_ml_features",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "cf80d032-4dde-4859-8ca4-bef9809698ab"},  # speed_limit
                    {"id": "9f74c5db-8288-4a99-99f9-9cbf668668b6"},  # light_conditions
                    {"id": "d29ae4fd-be94-4266-942f-45e352047b4e"},  # weather_conditions
                    {"id": "b2fed056-b67d-45e2-b2f0-dc7ea633082e"},  # road_surface_conditions
                    {"id": "5ec8f59d-2ef1-48b5-8e38-b50f85d467aa"},  # road_type
                    {"id": "0457c541-f506-40d2-8391-d1aeadd2cabf"},  # urban_or_rural_id
                    {"id": "85a63632-308f-49eb-a102-e42dfa6cf7cf"},  # number_of_vehicles
                    {"id": "c04d9172-10d8-411e-b3b9-c5c1cb58fa66"},  # number_of_casualties
                    {"id": "5c36efa9-6fee-45f5-9c11-62e6d021751c"},  # day_of_week
                    {"id": "b386f985-e1f0-4eb2-9f6c-48c3757d1623"},  # junction_detail
                    {"id": "d337b851-b297-45d8-bac9-671484c7cc76"},  # junction_control
                    {"id": "fe89f85f-34c3-47ac-a4d9-ac63ecc2f219"},  # pedestrian_crossing
                    {"id": "1de8034f-ee3f-43a7-af24-8e6167f6719a"},  # first_road_class
                    {"id": "552c3ecb-1f72-4d26-a53e-c769ea40a9bc"},  # special_conditions_at_site
                    {"id": "ce43f4aa-08f4-40f8-863b-05a5c3a1478f"},  # carriageway_hazards
                    {"id": "b1b75025-d546-4241-9ebc-bd497b0b07d4"},  # collision_severity
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    },
    {
        "name": "collision_severity_summary",
        "purpose": "Aggregated collision counts grouped by severity, road type, urban/rural area and speed limit. Used to verify class imbalance before SMOTE balancing in 02_preprocess.py.",
        "body": {
            "name": "collision_severity_summary",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "b1b75025-d546-4241-9ebc-bd497b0b07d4"},  # collision_severity
                    {"id": "0457c541-f506-40d2-8391-d1aeadd2cabf"},  # urban_or_rural_id
                    {"id": "5ec8f59d-2ef1-48b5-8e38-b50f85d467aa"},  # road_type
                    {"id": "cf80d032-4dde-4859-8ca4-bef9809698ab"},  # speed_limit
                    {"id": "c04d9172-10d8-411e-b3b9-c5c1cb58fa66"},  # number_of_casualties
                    {"id": "85a63632-308f-49eb-a102-e42dfa6cf7cf"},  # number_of_vehicles
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    },
    {
        "name": "collision_feature_stats",
        "purpose": "Descriptive statistics for numeric features: speed_limit, number_of_vehicles, number_of_casualties. Used to verify data integrity after API loading in T2.6.",
        "body": {
            "name": "collision_feature_stats",
            "is_public": True,
            "is_schema_public": True,
            "query": {
                "datasource_ids": [TABLE_ID],
                "columns": [
                    {"id": "b1b75025-d546-4241-9ebc-bd497b0b07d4"},  # collision_severity
                    {"id": "cf80d032-4dde-4859-8ca4-bef9809698ab"},  # speed_limit
                    {"id": "85a63632-308f-49eb-a102-e42dfa6cf7cf"},  # number_of_vehicles
                    {"id": "c04d9172-10d8-411e-b3b9-c5c1cb58fa66"},  # number_of_casualties
                ],
                "joins": [],
                "filters": [],
                "orders": []
            }
        }
    }
]

print(f"Defined {len(VIEWS)} views and ready to create")
for v in VIEWS:
    print(f"  - {v['name']}: {v['purpose'][:60]}...")

Defined 3 views and ready to create
  - collision_ml_features: Clean feature table matching exactly the input expected by 0...
  - collision_severity_summary: Aggregated collision counts grouped by severity, road type, ...
  - collision_feature_stats: Descriptive statistics for numeric features: speed_limit, nu...


In [10]:
created_views = {}

for view in VIEWS:
    print(f"Creating view: {view['name']}")

    response = requests.post(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json"
        },
        json=view["body"]
    )

    print(f"  Status: {response.status_code}")

    if response.status_code in (200, 201):
        view_id = response.json().get("id")
        created_views[view["name"]] = view_id
        print(f"  Created successfully with ID: {view_id}")
    else:
        print(f"  Failed: {response.text[:300]}")

print(f"\nTotal views created: {len(created_views)} out of {len(VIEWS)}")

Creating view: collision_ml_features
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}
Creating view: collision_severity_summary
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}
Creating view: collision_feature_stats
  Status: 403
  Failed: {"status":"FORBIDDEN","message":"Failed to create view: not the database owner","code":"error.request.forbidden"}

Total views created: 0 out of 3


In [11]:
response = requests.get(
    f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view",
    auth=HTTPBasicAuth(USERNAME, PASSWORD),
    headers={"Accept": "application/json"}
)

if response.status_code == 200:
    views_in_db = response.json()
    print(f"Views found in database: {len(views_in_db)}")
    for v in views_in_db:
        print(f"  ID: {v.get('id')} | Name: {v.get('name')}")
else:
    print(f"Failed with status {response.status_code}: {response.text[:200]}")

Views found in database: 2
  ID: 8ecff635-0597-447e-b59f-2b377a9cff0f | Name: collision_severity_summary
  ID: 1a8e1df5-c381-42fd-af08-2060a3e3ef94 | Name: collision_ml_features


In [13]:
if created_views.get("collision_ml_features"):
    view_id = created_views["collision_ml_features"]

    response = requests.get(
        f"{BASE_URL}/api/v1/database/{DATABASE_ID}/view/{view_id}/data",
        auth=HTTPBasicAuth(USERNAME, PASSWORD),
        headers={"Accept": "application/json"},
        params={"page": 0, "size": 5}
    )

    print(f"collision_ml_features spot check - Status: {response.status_code}")

    if response.status_code == 200:
        import pandas as pd
        data = response.json()
        df = pd.DataFrame(data)
        print(f"Rows returned: {len(df)}")
        print(f"Columns: {list(df.columns)}")
        print(df.head())
    else:
        print(f"Failed: {response.text[:200]}")
else:
    print("collision_ml_features was not created, skipping spot check")

collision_ml_features was not created, skipping spot check
